# LlamaIndex agents with deterministic tool selection

An original, deterministic LlamaIndex companion lesson.

## Learning objectives

- Describe how an agent selects and invokes a FunctionTool-like object.
- Keep tool metadata separate from the callable implementation.
- Produce a schedule answer grounded in a deterministic tool result.

## Setup

The offline agent loop uses only Python dictionaries and functions. It needs no model, network, or credential. The live cell maps the same design to a LlamaIndex `ReActAgent`, which supports models without native function calling.


The optional live example uses `python -m pip install llama-index llama-index-llms-huggingface-api`. Optional Phoenix tracing uses `python -m pip install arize-phoenix-otel`.

## Complete implementation

Run the next cell to build the complete offline demonstration.

In [1]:
schedule = {
    "gala": {"start": "19:00", "venue": "Grand Hall"},
    "workshop": {"start": "14:00", "venue": "Studio B"},
}

def lookup_schedule(event_name):
    """Return the embedded schedule entry for an event."""
    normalized_name = event_name.strip().casefold()
    if normalized_name not in schedule:
        return {"error": "event not found"}
    return schedule[normalized_name]

function_tools = [
    {
        "name": "lookup_schedule",
        "description": "Look up the start time and venue for a named event.",
        "required_argument": "event_name",
        "callable": lookup_schedule,
    }
]

agent_request = "When does the gala start?"
selected_tool = next(
    tool for tool in function_tools if "gala" in agent_request.casefold()
)
tool_arguments = {"event_name": "gala"}
schedule_observation = selected_tool["callable"](**tool_arguments)
agent_trace = [
    {"stage": "select", "tool": selected_tool["name"]},
    {"stage": "call", "arguments": tool_arguments},
    {"stage": "observe", "result": schedule_observation},
]
agent_answer = f"The gala starts at {schedule_observation['start']}."

## Deterministic assertions

Run these checks to verify the lesson without credentials, network access, or third-party packages.

In [2]:
assert selected_tool["name"] == "lookup_schedule"
assert tool_arguments == {"event_name": "gala"}
assert schedule_observation == {"start": "19:00", "venue": "Grand Hall"}
assert [entry["stage"] for entry in agent_trace] == ["select", "call", "observe"]
assert agent_answer == "The gala starts at 19:00."

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

The natural-language request is converted into an explicit tool choice and structured arguments. Because the final sentence uses the tool observation rather than hidden knowledge, the answer is inspectable and reproducible.

## Credentials

Offline cells use no credentials. The live cell checks `HF_TOKEN` before importing LlamaIndex and never displays the token. `PHOENIX_API_KEY` is checked separately before Phoenix is imported. Google or Gmail tools require a separate OAuth client and interactive consent flow; this notebook neither requests nor embeds OAuth material.

In [3]:
import os

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    print("SKIPPED: Set HF_TOKEN to run this optional LlamaIndex example.")
else:
    try:
        from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
        from llama_index.core.agent.workflow import ReActAgent
        from llama_index.core.tools import FunctionTool
    except ImportError:
        print("SKIPPED: llama-index is not installed; run `python -m pip install llama-index llama-index-llms-huggingface-api`.")
    else:
        phoenix_api_key = os.getenv("PHOENIX_API_KEY")
        if phoenix_api_key:
            try:
                from phoenix.otel import register
            except ImportError:
                print("SKIPPED: Phoenix tracing is not installed; run `python -m pip install arize-phoenix-otel`.")
            else:
                tracer_provider = register(
                    project_name="hf-agents-course-llama-index",
                    endpoint="https://app.phoenix.arize.com/v1/traces",
                    headers={"api_key": phoenix_api_key},
                    auto_instrument=True,
                )
                print("Phoenix tracing configured.")
        else:
            print("SKIPPED: PHOENIX_API_KEY is absent; tracing remains disabled.")

        llm = HuggingFaceInferenceAPI(
            model_name="Qwen/Qwen2.5-7B-Instruct",
            token=hf_token,
            task="text-generation",
        )
        def live_lookup_schedule(event_name: str) -> str:
            """Return a known start time for an event name."""
            return {"gala": "19:00", "workshop": "14:00"}.get(
                event_name.casefold(), "unknown"
            )

        schedule_tool = FunctionTool.from_defaults(fn=live_lookup_schedule)
        live_agent = ReActAgent(
            tools=[schedule_tool],
            llm=llm,
            system_prompt="Use the schedule tool for event-time questions.",
        )
        print("Configured a live ReActAgent; call `await live_agent.run(...)` in an async notebook context.")

SKIPPED: PHOENIX_API_KEY is absent; tracing remains disabled.
Configured a live ReActAgent; call `await live_agent.run(...)` in an async notebook context.


## Completion checklist

- [ ] Inspected the FunctionTool-like metadata.
- [ ] Followed the select, call, and observe trace.
- [ ] Verified that the answer uses the schedule observation.

## Attribution

This example is newly authored for the course companion. Its interface is informed by the official [LlamaIndex agents documentation](https://docs.llamaindex.ai/en/stable/module_guides/deploying/agents/) and [tools documentation](https://docs.llamaindex.ai/en/stable/module_guides/deploying/agents/tools/); no upstream notebook text or code is copied. Notebook code carries the Apache-2.0 metadata license.